# T2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import seaborn as sns
import math as mt
from skimage.io import imread, imsave
import statsmodels.api as sm


np.random.seed(42)
plt.style.use('ggplot')

In [3]:
X = np.array([[83, 85],
              [84, 85, 85, 86, 86, 87],
              [86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90],
              [89, 90, 90, 91],
              [90, 92]], dtype=object)

a) Определить влияние возраста на содержание имунноглобулина в крови с помощью регрессионного анализа

In [4]:
lens = np.array([len(x) for x in X])
Y = []
for i in range(len(X)):
    for j in range(lens[i]):
        Y+=[X[i][j]]
n = len(Y)
p = len(X)
alpha = 0.05 
total_len = np.sum(lens)
PSI = np.array([np.zeros(5) for i in range(total_len)])
j_total = 0
for i in range(len(X)):
    j = 0
    while j < lens[i]:
        PSI[j_total + j,i]=1
        j+=1
    j_total += lens[i]
F = PSI.T @ PSI
F_inv = np.linalg.inv(F)
betas = F_inv @ PSI.T @ Y
print('Уравнение линейной регрессии:')
print(f'y = {betas[0]:.3f} * x_1 + {betas[1]:.3f} * x_2 + {betas[2]:.3f} * x_3 + {betas[3]:.3f} * x_4 + {betas[4]:.3f} * x_5 + e')
errors = Y - PSI @ betas
RSS = errors.T @ errors 
TSS = np.sum((Y - np.mean(Y))**2)
R_squared = 1 - RSS/TSS
print(f'R^2: {R_squared:.4f}')
print('Проверим значимость коэффициентов регрессии:')
for i in range(0, len(betas)):
    delt_T = abs( (betas[i] * np.sqrt(n-p)) / (np.sqrt(RSS * F_inv[i,i])) ) # при верности H0 beta истинная должна быть равна нулю (незначима)
    p_value = 2 * (1 - st.t.cdf(delt_T, df= (n - p)))
    print(f'Коэффициент при x_{i}: статистика критерия = {delt_T:.3f}, p_value = {p_value:.3f}, ', 'Коэффициент значим' if p_value < alpha else 'Коэффициент не значим')

Уравнение линейной регрессии:
y = 84.000 * x_1 + 85.500 * x_2 + 87.818 * x_3 + 90.000 * x_4 + 91.000 * x_5 + e
R^2: 0.8106
Проверим значимость коэффициентов регрессии:
Коэффициент при x_0: статистика критерия = 110.449, p_value = 0.000,  Коэффициент значим
Коэффициент при x_1: статистика критерия = 194.719, p_value = 0.000,  Коэффициент значим
Коэффициент при x_2: статистика критерия = 270.800, p_value = 0.000,  Коэффициент значим
Коэффициент при x_3: статистика критерия = 167.355, p_value = 0.000,  Коэффициент значим
Коэффициент при x_4: статистика критерия = 119.653, p_value = 0.000,  Коэффициент значим


b) Провести попарное сравнение средних в рамках регрессионной модели с учетом множественности проверяемых гипотез

In [5]:
def delt_T(betas, i, j):
    return abs( (betas[i] - betas[j]) * np.sqrt(n-p) / np.sqrt(RSS * (F_inv[i,i] + F_inv[j,j] - 2*F_inv[i,j])) )

def p_val(delta_T):
    return 2 * (1 - st.t(n-p).cdf(delta_T))


print('Проведем попарное сравнение коэффициентов регрессии:')
df = pd.DataFrame(columns=['betas', 'delta', 'p-value', 'conclusion'])
pvals = []
deltas = []
conclusions = []
betas_names = []
for i in range(0, len(betas)):
    for j in range(i+1, len(betas)):
        delta_T = delt_T(betas, i, j)
        pval = p_val(delta_T)
        betas_names.append(f'beta_{i} и beta_{j}')
        deltas.append(delta_T)
        pvals.append(pval)
        conclusions.append('Значимо отличаются' if pval < alpha else 'Не значимо') 
df['betas'] = betas_names
df['delta'] = deltas
df['p-value'] = pvals
df['conclusion'] = conclusions
print('Краткий итог:')
print(df)

Проведем попарное сравнение коэффициентов регрессии:
Краткий итог:
             betas     delta   p-value          conclusion
0  beta_0 и beta_1  1.708065  0.103100          Не значимо
1  beta_0 и beta_2  4.618104  0.000166  Значимо отличаются
2  beta_0 и beta_3  6.441516  0.000003  Значимо отличаются
3  beta_0 и beta_4  6.508269  0.000002  Значимо отличаются
4  beta_1 и beta_2  4.246806  0.000395  Значимо отличаются
5  beta_1 и beta_3  6.481650  0.000003  Значимо отличаются
6  beta_1 и beta_4  6.262904  0.000004  Значимо отличаются
7  beta_2 и beta_3  3.474295  0.002393  Значимо отличаются
8  beta_2 и beta_4  3.848420  0.001003  Значимо отличаются
9  beta_3 и beta_4  1.073586  0.295791          Не значимо
